<a href="https://colab.research.google.com/github/rahmanullahkhan123/Generative_AI/blob/main/RUK_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.11.0+cpu
GPU Available: False


In [18]:
!pip install tiktoken transformers datasets
!pip install gradio
!pip install tokenizers gradio

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

import gradio as gr

In [20]:
text = """

Artificial Intelligence is the future of technology.

Machine Learning allows computers to learn from data.

Deep Learning uses artificial neural networks.

Natural Language Processing helps computers understand human language.

Transformers are powerful deep learning architectures.

GPT models generate human like text.

Large Language Models learn patterns from huge datasets.

RUK_AI is a custom Mini GPT model.

RUK_AI can generate text and answer questions.

Generative AI creates new content using neural networks.

"""


with open(
    "train.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(text)


print("Dataset Created")

Dataset Created


In [21]:
tokenizer = Tokenizer(
    BPE()
)


tokenizer.pre_tokenizer = Whitespace()


trainer = BpeTrainer(
    vocab_size=1000,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[BOS]",
        "[EOS]"
    ]
)


tokenizer.train(
    ["train.txt"],
    trainer
)


tokenizer.save(
    "ruk_ai_tokenizer.json"
)


print(
    "Vocabulary:",
    tokenizer.get_vocab_size()
)

Vocabulary: 233


In [22]:
tokenizer = Tokenizer.from_file(
    "ruk_ai_tokenizer.json"
)


vocab_size = tokenizer.get_vocab_size()


print(
    "Vocab Size:",
    vocab_size
)

Vocab Size: 233


In [23]:
with open(
    "train.txt",
    encoding="utf-8"
) as f:
    text=f.read()



encoded = tokenizer.encode(text)


data = torch.tensor(
    encoded.ids,
    dtype=torch.long
)


print(data)
print(data.shape)

tensor([201, 228,  87, 168, 224, 156, 220,   4, 206, 113, 223, 122, 166,  63,
        118, 117,   4, 202, 113, 188, 177, 104, 119,   4, 209,  76, 229, 225,
        122, 219, 108, 149,   4, 230, 175, 217, 182, 186, 222,   4, 111, 195,
        120, 108, 193, 101,   4, 205,  76, 208,  63, 216, 118, 185, 226,   4,
        114,  87,  18, 214, 207, 111,  92,   4, 114, 137, 120, 101, 102, 197,
        232,   4, 227,  57, 231, 178, 212, 187, 104, 119,   4])
torch.Size([81])


In [24]:
class SelfAttention(nn.Module):

    def __init__(self, embed_size):

        super().__init__()

        self.query = nn.Linear(
            embed_size,
            embed_size
        )

        self.key = nn.Linear(
            embed_size,
            embed_size
        )

        self.value = nn.Linear(
            embed_size,
            embed_size
        )


    def forward(self,x):

        Q=self.query(x)

        K=self.key(x)

        V=self.value(x)


        attention = torch.matmul(
            Q,
            K.transpose(-2,-1)
        )


        attention = attention / (
            x.shape[-1] ** 0.5
        )


        attention = F.softmax(
            attention,
            dim=-1
        )


        output=torch.matmul(
            attention,
            V
        )


        return output

In [25]:
class GPTBlock(nn.Module):

    def __init__(self, embed):

        super().__init__()


        self.attention = SelfAttention(
            embed
        )


        self.norm1 = nn.LayerNorm(
            embed
        )


        self.feedforward = nn.Sequential(

            nn.Linear(
                embed,
                embed*4
            ),

            nn.ReLU(),

            nn.Linear(
                embed*4,
                embed
            )
        )


        self.norm2 = nn.LayerNorm(
            embed
        )


    def forward(self,x):

        x = self.norm1(
            x + self.attention(x)
        )


        x = self.norm2(
            x + self.feedforward(x)
        )


        return x

In [26]:
class RUK_AI_GPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_size=128
    ):

        super().__init__()


        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_size
        )


        self.position_embedding = nn.Embedding(
            256,
            embed_size
        )


        self.blocks = nn.Sequential(

            GPTBlock(embed_size),

            GPTBlock(embed_size),

            GPTBlock(embed_size)

        )


        self.output = nn.Linear(
            embed_size,
            vocab_size
        )


    def forward(self,x):

        B,T=x.shape


        token=self.token_embedding(x)


        position=self.position_embedding(
            torch.arange(T).to(device)
        )


        x=token+position


        x=self.blocks(x)


        logits=self.output(x)


        return logits

In [27]:
model = RUK_AI_GPT(
    vocab_size
)


model.to(device)


print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

Parameters: 637929


In [28]:
optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)


model.train()


for epoch in range(500):

    x=data[:-1].unsqueeze(0).to(device)

    y=data[1:].unsqueeze(0).to(device)



    output=model(x)


    loss=F.cross_entropy(
        output.reshape(-1,vocab_size),
        y.reshape(-1)
    )


    optimizer.zero_grad()

    loss.backward()

    optimizer.step()



    if epoch%50==0:

        print(
            "Epoch:",
            epoch,
            "Loss:",
            loss.item()
        )

Epoch: 0 Loss: 5.638295650482178
Epoch: 50 Loss: 0.03191875293850899
Epoch: 100 Loss: 0.013182850554585457
Epoch: 150 Loss: 0.008101141080260277
Epoch: 200 Loss: 0.005563344806432724
Epoch: 250 Loss: 0.004089524503797293
Epoch: 300 Loss: 0.0031508938409388065
Epoch: 350 Loss: 0.0025128170382231474
Epoch: 400 Loss: 0.002057492733001709
Epoch: 450 Loss: 0.0017200408037751913


In [29]:
torch.save(
    model.state_dict(),
    "RUK_AI_v2.pth"
)


print("Model Saved")

Model Saved


In [30]:
def generate_text(prompt):

    model.eval()


    tokens = tokenizer.encode(prompt).ids


    x=torch.tensor(
        tokens
    ).unsqueeze(0).to(device)



    with torch.no_grad():

        output=model(x)



    prediction=torch.argmax(
        output,
        dim=-1
    )


    result=tokenizer.decode(
        prediction[0].cpu().numpy()
    )


    return result

In [31]:
generate_text(
    "Artificial Intelligence"
)

'Intelligence is'

In [32]:
def ruk_ai_chat(message, history):

    response = generate_text(
        message
    )

    return response



demo = gr.ChatInterface(
    fn=ruk_ai_chat,
    title="🤖 RUK_AI v2 Mini GPT",
    description="""
    Custom GPT-style Language Model

    Features:
    ✅ BPE Tokenizer
    ✅ Self Attention
    ✅ Transformer Blocks
    ✅ Text Generation
    """
)


demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed945c811b71839607.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
